In [69]:
from __future__ import annotations

%load_ext autoreload

%autoreload 2
from typing import TYPE_CHECKING, Any, cast

import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from funcs.data_import import describe_data, import_data
from funcs.pipeline import build_preprocessing_pipeline

if TYPE_CHECKING:
    from numpy.typing import NDArray


pl.Config.set_tbl_cols(-1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


polars.config.Config

In [70]:
train_df: pl.DataFrame = import_data("data/train.csv")
test_df: pl.DataFrame = import_data("data/test.csv")

In [ ]:
describe_data(
    train_df,
    [
        "price",
        "mileage",
        "tax",
        "mpg",
        "engineSize",
        "paintQuality%",
        "previousOwners",
    ],
    ["Brand", "model", "year", "transmission", "fuelType", "hasDamage"],
)

In [77]:
preproc_pipeline: Pipeline = build_preprocessing_pipeline(
    metric_features=[
        "mileage",
        "tax",
        "mpg",
        "engineSize",
        "paintQuality%",
        "previousOwners",
    ],
    bool_features=[
        "hasDamage",
    ],
    categorical_features=[
        "Brand",
        "model",
        "year",
        "transmission",
        "fuelType",
    ],
    thresholds={
        "mileage": {"lower": 0, "upper": 200000},
        "tax": {"lower": 0, "upper": 350},
        "mpg": {"lower": 20, "upper": 150},
        "engineSize": {"lower": 1.0, "upper": 4.5},
        "paintQuality%": {"lower": 0, "upper": 100},
        "previousOwners": {"lower": 0, "upper": 5},
    },
    winsorize=True,
    unneeded_float_features=[
        "year",
        "mileage",
        "tax",
        "paintQuality%",
        "previousOwners",
        "hasDamage",
    ],
    scaling_exclude_selector=("_", "hasDamage", "carID"),
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", preproc_pipeline),
        # ("feature_selection", SelectKBest(f_regression, k=100)),
        ("lr", LinearRegression()),
    ]
)

pipe_x: pl.DataFrame = train_df.drop("price")
pipe_y: pl.DataFrame = train_df.select("price")

pipe_x_train: pl.DataFrame
pipe_x_val: pl.DataFrame
pipe_y_train: pl.Series
pipe_y_val: pl.Series

pipe_x_train, pipe_x_val, pipe_y_train, pipe_y_val = train_test_split(
    pipe_x, pipe_y, test_size=0.2, random_state=42
)

pipeline.fit(pipe_x_train, pipe_y_train)
train_score: float = cast("float", pipeline.score(pipe_x_train, pipe_y_train))
val_score: float = cast("float", pipeline.score(pipe_x_val, pipe_y_val))
print(f"Train score: {train_score}")
print(f"Validation score: {val_score}")
print("Score difference:", train_score - val_score)

Train score: 0.8571381210715457
Validation score: 0.8653690711097405
Score difference: -0.00823095003819474


In [72]:
predictions: NDArray[Any] = cast(
    "NDArray[Any]", pipeline.predict(test_df)
).flatten()

pl.DataFrame(
    {"carID": test_df.get_column("carID"), "price": predictions}
).write_csv("data/submission.csv")